# `farmland-mpc` — Local Container Notebook (A → B → C → D)

Drop a DLTB shapefile into `/work/in/` on the host (the directory you mounted with `-v`), set the cell below, then **Run All**.

This is the local-container twin of `farmland_mpc_colab_full.ipynb`. Same algorithm, same output. The only difference: no Google Drive mount; `/work` is the volume the host mapped in.

In [ ]:
from pathlib import Path

# === EDIT THESE ===
INPUT_DLTB = Path('/work/in/DLTB.shp')   # your shapefile inside the mounted volume
WORK_DIR   = Path('/work/run_1')         # outputs land here
PROJ_CRS   = None                         # None = auto-pick UTM zone from centroid

WORK_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_DLTB.exists(), f'Not found: {INPUT_DLTB}'
for ext in ('.shx', '.dbf', '.prj'):
    assert INPUT_DLTB.with_suffix(ext).exists(), f'Missing: {INPUT_DLTB.with_suffix(ext)}'
print(f'INPUT_DLTB: {INPUT_DLTB}')
print(f'WORK_DIR:   {WORK_DIR}')

## 1 · Inspect shapefile and pick CRS

In [ ]:
import geopandas as gpd

dltb = gpd.read_file(INPUT_DLTB)
print(f'parcels:  {len(dltb):,}')
print(f'CRS:      {dltb.crs}')
print(f'columns:  {list(dltb.columns)[:12]}...')
for col in ('DLBM', 'QSDWDM', 'BSM'):
    if col not in dltb.columns:
        print(f'  WARN: required column {col!r} missing')

if PROJ_CRS is None:
    cen = dltb.to_crs('EPSG:4326').geometry.union_all().centroid
    zone = int((cen.x + 180) // 6) + 1
    PROJ_CRS = f'EPSG:{(32600 if cen.y >= 0 else 32700) + zone}'
    print(f'auto-picked CRS: {PROJ_CRS}')

## 2 · Fetch DEM (calls scripts/fetch_dem.py — needs out-going access to AWS S3)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '/repo/scripts/fetch_dem.py',
                '--dltb', str(INPUT_DLTB),
                '--work-dir', str(WORK_DIR),
                '--proj-crs', PROJ_CRS], check=True)
DEM_FINAL = WORK_DIR / 'dem.tif'
print(f'DEM ready: {DEM_FINAL}  ({DEM_FINAL.stat().st_size/1e6:.1f} MB)')

## 3 · Phase A: prepare (slope + blocks)

In [ ]:
from farmland_mpc.prepare import run as prepare_run

PREPARED = WORK_DIR / 'prepared'
prepare_run(
    dltb_path=str(INPUT_DLTB),
    dem_path=str(DEM_FINAL),
    prepared_dir=str(PREPARED),
    proj_crs=PROJ_CRS,
)

## 4 · Phase B: sample (~19 min on M-series, ~12 min on i7-13700K)

In [ ]:
from farmland_mpc.sample import run as sample_run
sample_run(prepared_dir=str(PREPARED),
           n_transition_episodes=60,
           n_pairwise_states=1000,
           n_pairwise_actions=50,
           seed=0,
           proj_crs=PROJ_CRS)

## 5 · Phase C: train (3 × 30 epochs, ~37 min on M-series)

In [ ]:
from farmland_mpc.train_ensemble import run as train_run
train_run(prepared_dir=str(PREPARED), n_members=3, epochs=30,
          patience=8, lambda_rank=5.0, margin=0.1,
          batch_size=256, n_pairs_per_state=10, pw_subsample=100,
          lr=1e-3, weight_decay=1e-5, seed_base=0, torch_threads=0)

## 6 · Phase D: MPC plan (~5 min/episode)

In [ ]:
from farmland_mpc.mpc_plan import run as plan_run
MPC_OUT = WORK_DIR / 'mpc_output'
MPC_OUT.mkdir(exist_ok=True)
plan_run(ensemble_dir=str(PREPARED / 'tool3'),
         out_dir=str(MPC_OUT),
         prepared_dir=str(PREPARED),
         proj_crs=PROJ_CRS,
         horizon=5, top_k=50, n_episodes=5,
         continuation='greedy', scoring='reward',
         seed_offset=0,
         output_fc=str(MPC_OUT / 'optimized.shp'),
         input_dltb_fc=str(INPUT_DLTB))
print(f'\nDone. Find optimized.shp at: {MPC_OUT}')